In [ ]:
!pip install -q litellm jsonschema

In [ ]:
import os, json, re
from dataclasses import dataclass, field
from google.colab import userdata
from litellm import completion
from jsonschema import validate

# Key from Colab Secrets (left sidebar > key icon > add GEMINI_API_KEY, toggle "Notebook access" on)
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

MODEL = "gemini/gemini-3.6-flash"  # change if your model ID differs


# Minimal stand-ins so your original function shape works unchanged
@dataclass
class ActionContext:
  # 
    metadata: dict = field(default_factory=dict)

def register_tool(tags=None):
    def decorator(fn):
        fn.tags = tags or []
        return fn
    return decorator


def generate_response(messages, model=MODEL, response_format=None, temperature=0.0):
    """Thin wrapper around litellm.completion that returns the text."""
    kwargs = dict(model=model, messages=messages, temperature=temperature)
    if response_format:
        kwargs["response_format"] = response_format
    resp = completion(**kwargs)
    return resp.choices[0].message.content


def _parse_json(text: str) -> dict:
    """Parse JSON even if the model wrapped it in ```json fences."""
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text)
    return json.loads(text)


def prompt_llm_for_json(action_context: ActionContext, schema: dict, prompt: str) -> dict:
    """Ask the LLM for JSON that conforms to `schema`, then validate it."""
    system = (
        "You are a precise data extraction engine. Respond ONLY with valid JSON "
        "that conforms to this JSON schema. Use null-free output: omit fields "
        "you cannot find (except required ones).\n\n"
        f"Schema:\n{json.dumps(schema, indent=2)}"
    )
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": prompt},
    ]

    try:
        raw = generate_response(
            messages,
            response_format={"type": "json_object"},
        )
    except Exception:
        # Fallback if the provider rejects response_format
        raw = generate_response(messages)

    data = _parse_json(raw)
    validate(instance=data, schema=schema)  # raises if required fields are missing
    return data

In [ ]:
@register_tool(tags=["document_processing", "invoices"])
def extract_invoice_data(action_context: ActionContext, document_text: str) -> dict:
    """Extract standardized invoice data from document text."""
    invoice_schema = {
        "type": "object",
        "required": ["invoice_number", "date", "amount"],
        "properties": {
            "invoice_number": {"type": "string"},
            "date": {"type": "string", "description": "ISO format YYYY-MM-DD"},
            "amount": {
                "type": "object",
                "properties": {
                    "value": {"type": "number"},
                    "currency": {"type": "string"},
                },
                "required": ["value", "currency"],
            },
            "vendor": {
                "type": "object",
                "properties": {
                    "name": {"type": "string"},
                    "tax_id": {"type": "string"},
                    "address": {"type": "string"},
                },
                "required": ["name"],
            },
            "line_items": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "description": {"type": "string"},
                        "quantity": {"type": "number"},
                        "unit_price": {"type": "number"},
                        "total": {"type": "number"},
                    },
                    "required": ["description", "total"],
                },
            },
        },
    }

    extraction_prompt = f"""
    Extract invoice information from the following document text.
    Focus on identifying:
    - Invoice number (usually labeled as 'Invoice #', 'Reference', etc.)
    - Date (any dates labeled as 'Invoice Date', 'Issue Date', etc.), formatted YYYY-MM-DD
    - Amount (total amount due, including currency)
    - Vendor information (company name, tax ID if present, address)
    - Line items (individual charges and their details)

    Document text:
    {document_text}
    """

    return prompt_llm_for_json(
        action_context=action_context,
        schema=invoice_schema,
        prompt=extraction_prompt,
    )

In [ ]:
sample_invoice = """
ACME Supplies Ltd
12 Marina Road, Lagos, Nigeria
Tax ID: NG-123456789

INVOICE #INV-2026-0042
Invoice Date: October 1, 2026

Description            Qty   Unit Price   Total
Office chairs           4     45,000      180,000
Standing desk           2     120,000     240,000
Delivery                1     15,000      15,000

Total Due: NGN 435,000
"""

result = extract_invoice_data(ActionContext(), sample_invoice)
print(json.dumps(result, indent=2))